# Statistical Modeling

## What Predicts if a Flight Will be Delayed?

In [4]:
import pandas as pd
import duckdb as db
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

PATH = "clean_flights.parquet"

In [2]:
con = db.connect()

### Logistic Regression

In [3]:
model_df = con.execute(f"""
SELECT
    CASE 
        WHEN GREATEST(DepDelayMinutes, ArrDelayMinutes) > 15 THEN 1
        ELSE 0
    END AS IsDelayed,

    TaxiOut,
    Distance,
    Month,
    FLOOR(CAST(CRSDepTime AS INTEGER)/100) AS DepHour,
    Reporting_Airline,
    Origin

FROM read_parquet('{PATH}')
USING SAMPLE 200000 ROWS
""").df()

In [5]:
X = model_df.drop("IsDelayed", axis=1)
y = model_df["IsDelayed"]

categorical_cols = ["Reporting_Airline", "Origin"]
numeric_cols = ["TaxiOut", "Distance", "Month", "DepHour"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols),
    ],
    remainder="drop"
)

clf = LogisticRegression(
    solver="saga",
    max_iter=5000,
    n_jobs=-1,
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", clf)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]
preds = (probs >= 0.5).astype(int)

print("AUC:", roc_auc_score(y_test, probs))
print(classification_report(y_test, preds, digits=3))

/Users/hudsonpifer/CSC369/assignments/assignment1/FinalProject/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


AUC: 0.6802641907360706
              precision    recall  f1-score   support

           0      0.795     0.986     0.880     31133
           1      0.682     0.106     0.184      8867

    accuracy                          0.791     40000
   macro avg      0.738     0.546     0.532     40000
weighted avg      0.770     0.791     0.726     40000



## Logistic Regression with P-Values

### Are Month and Delay independent?

In [6]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# contingency table
ct = pd.crosstab(model_df["Month"], model_df["IsDelayed"])

chi2, p, dof, expected = chi2_contingency(ct)

n = ct.to_numpy().sum()
k = min(ct.shape)  # min(#rows, #cols)
cramers_v = np.sqrt(chi2 / (n * (k - 1)))

summary = pd.DataFrame({
    "chi2": [chi2],
    "dof": [dof],
    "p_value": [p],
    "cramers_v": [cramers_v],
    "n": [n],
})

display(summary)

# Month to month delay rate:
month_rates = (ct.div(ct.sum(axis=1), axis=0)
                 .rename(columns={0: "pct_not_delayed", 1: "pct_delayed"}))
month_rates["pct_delayed"] = month_rates["pct_delayed"] * 100
month_rates = month_rates[["pct_delayed"]].sort_values("pct_delayed", ascending=False)

display(month_rates)

,chi2,dof,p_value,cramers_v,n
0,1344.636378,11,1.056970e-281,0.081995,200000


IsDelayed,pct_delayed
Month,
7,28.890618
6,27.539110
8,23.699196
12,23.393895
5,23.236464
1,21.357444
4,21.034899
3,20.592179
2,20.580865


We strongly reject the null hypothesis that delay status is independent of month due to a p-value of almost 0.

The effect of month on delay probability is statistically significant but modest in magnitude since Cramer's V = 0.082.

### Are Airport and Delay independent?

In [7]:
top_airports = (
    model_df["Origin"]
    .value_counts()
    .head(30)
    .index
)

airport_df = model_df[model_df["Origin"].isin(top_airports)]

ct_airport = pd.crosstab(airport_df["Origin"], airport_df["IsDelayed"])

chi2, p, dof, expected = chi2_contingency(ct_airport)

n = ct_airport.to_numpy().sum()
k = min(ct_airport.shape)
cramers_v = np.sqrt(chi2 / (n * (k - 1)))

pd.DataFrame({
    "chi2": [chi2],
    "dof": [dof],
    "p_value": [p],
    "cramers_v": [cramers_v],
    "n": [n],
})

,chi2,dof,p_value,cramers_v,n
0,623.344661,29,9.184403e-113,0.069357,129582


### Are Airline and Delay Independent?

In [8]:
ct_airline = pd.crosstab(model_df["Reporting_Airline"], model_df["IsDelayed"])

chi2, p, dof, expected = chi2_contingency(ct_airline)

n = ct_airline.to_numpy().sum()
k = min(ct_airline.shape)
cramers_v = np.sqrt(chi2 / (n * (k - 1)))

pd.DataFrame({
    "chi2": [chi2],
    "dof": [dof],
    "p_value": [p],
    "cramers_v": [cramers_v],
    "n": [n],
})

,chi2,dof,p_value,cramers_v,n
0,1730.644956,17,0.0,0.093023,200000


### Are weather presence and sever delay independent?


In [9]:
severity_df = con.execute(f"""
SELECT
    CASE
        WHEN GREATEST(DepDelayMinutes, ArrDelayMinutes) > 180 THEN 1
        ELSE 0
    END AS Severe,

    CASE
        WHEN WeatherDelay > 0 THEN 1
        ELSE 0
    END AS WeatherPresent

FROM read_parquet('{PATH}')
USING SAMPLE 200000 ROWS
""").df()

In [10]:
ct_weather_severe = pd.crosstab(
    severity_df["WeatherPresent"],
    severity_df["Severe"]
)

chi2, p, dof, expected = chi2_contingency(ct_weather_severe)

n = ct_weather_severe.to_numpy().sum()
k = min(ct_weather_severe.shape)
cramers_v = np.sqrt(chi2 / (n * (k - 1)))

pd.DataFrame({
    "chi2": [chi2],
    "dof": [dof],
    "p_value": [p],
    "cramers_v": [cramers_v],
    "n": [n],
})

,chi2,dof,p_value,cramers_v,n
0,2632.74184,1,0.0,0.114733,200000


### Effect Size Summary Table

In [11]:
effect_summary = pd.DataFrame({
    "Factor": [
        "Month → Delay",
        "Airport → Delay",
        "Airline → Delay",
        "Weather Presence → Severe Delay"
    ],
    "Cramers_V": [
        0.081995,   # Month
        0.069357,   # Airport
        0.093023,   # Airline
        0.114733    # Weather → Severe
    ]
}).sort_values("Cramers_V", ascending=False)

def interpret_v(v):
    if v < 0.10:
        return "Small"
    elif v < 0.30:
        return "Moderate"
    else:
        return "Large"

effect_summary["Effect_Size"] = effect_summary["Cramers_V"].apply(interpret_v)

effect_summary

,Factor,Cramers_V,Effect_Size
3,Weather Presence → Severe Delay,0.114733,Moderate
2,Airline → Delay,0.093023,Small
0,Month → Delay,0.081995,Small
1,Airport → Delay,0.069357,Small


## Interaction Effects

In [12]:
model_df_small = con.execute(f"""
SELECT
    CASE 
        WHEN GREATEST(DepDelayMinutes, ArrDelayMinutes) > 15 THEN 1
        ELSE 0
    END AS IsDelayed,

    TaxiOut,
    Distance,
    Month,
    FLOOR(CAST(CRSDepTime AS INTEGER)/100) AS DepHour,
    Reporting_Airline

FROM read_parquet('{PATH}')
USING SAMPLE 200000 ROWS
""").df()

In [13]:
import statsmodels.formula.api as smf

# Treat Month and Airline as categorical
model_df_small["Month"] = model_df_small["Month"].astype("category")
model_df_small["Reporting_Airline"] = model_df_small["Reporting_Airline"].astype("category")

logit_model = smf.logit(
    formula="IsDelayed ~ TaxiOut + Distance + C(Month) + DepHour + C(Reporting_Airline)",
    data=model_df_small
).fit()

print(logit_model.summary())

Optimization terminated successfully.
         Current function value: 0.482244
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:              IsDelayed   No. Observations:               200000
Model:                          Logit   Df Residuals:                   199968
Method:                           MLE   Df Model:                           31
Date:                Mon, 09 Mar 2026   Pseudo R-squ.:                 0.08436
Time:                        22:13:41   Log-Likelihood:                -96449.
converged:                       True   LL-Null:                   -1.0533e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept                     -4.3084      0.049    -87.106      0.000      

After controlling for airline and seasonal effects, departure hour and taxi-out time emerge as the strongest predictors of delay probability. Each additional hour later in the day increases the odds of delay by approximately 9%, while each additional minute of taxi-out increases delay odds by roughly 6%. Airline effects are also substantial, with some carriers exhibiting more than triple the baseline odds of delay.

In [14]:
interaction_model = smf.logit(
    formula="""
    IsDelayed ~ TaxiOut 
               + Distance 
               + C(Month) * DepHour 
               + C(Reporting_Airline)
    """,
    data=model_df_small
).fit()

print(interaction_model.summary())

Optimization terminated successfully.
         Current function value: 0.481663
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:              IsDelayed   No. Observations:               200000
Model:                          Logit   Df Residuals:                   199957
Method:                           MLE   Df Model:                           42
Date:                Mon, 09 Mar 2026   Pseudo R-squ.:                 0.08546
Time:                        22:19:33   Log-Likelihood:                -96333.
converged:                       True   LL-Null:                   -1.0533e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept                     -4.0462      0.073    -55.118      0.000      

The impact of departure hour on delay probability is significantly stronger during summer months. For example, in July the marginal effect of departure hour nearly doubles relative to baseline months, indicating that delay accumulation is seasonally amplified.

In [15]:
interaction_model2 = smf.logit(
    formula="""
    IsDelayed ~ Distance 
               + C(Month) * TaxiOut
               + DepHour 
               + C(Reporting_Airline)
    """,
    data=model_df_small
).fit()

print(interaction_model2.summary())

Optimization terminated successfully.
         Current function value: 0.482088
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:              IsDelayed   No. Observations:               200000
Model:                          Logit   Df Residuals:                   199957
Method:                           MLE   Df Model:                           42
Date:                Mon, 09 Mar 2026   Pseudo R-squ.:                 0.08465
Time:                        22:46:51   Log-Likelihood:                -96418.
converged:                       True   LL-Null:                   -1.0533e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept                     -4.3811      0.061    -71.944      0.000      

In [17]:
model_df_small = con.execute(f"""
SELECT
    CASE 
        WHEN GREATEST(DepDelayMinutes, ArrDelayMinutes) > 15 THEN 1
        ELSE 0
    END AS IsDelayed,

    CASE 
        WHEN WeatherDelay > 0 THEN 1
        ELSE 0
    END AS WeatherPresent,

    TaxiOut,
    Distance,
    Month,
    FLOOR(CAST(CRSDepTime AS INTEGER)/100) AS DepHour,
    Reporting_Airline,
    Origin

FROM read_parquet('{PATH}')
USING SAMPLE 200000 ROWS
""").df()

In [18]:
top_airports = (
    model_df_small["Origin"]
    .value_counts()
    .head(20)
    .index
)

model_df_small["LargeHub"] = (
    model_df_small["Origin"].isin(top_airports)
).astype(int)

In [19]:
import statsmodels.formula.api as smf

interaction_model3 = smf.logit(
    formula="""
    IsDelayed ~ TaxiOut
               + Distance
               + DepHour
               + WeatherPresent * LargeHub
               + C(Month)
               + C(Reporting_Airline)
    """,
    data=model_df_small
).fit()

print(interaction_model3.summary())

Optimization terminated successfully.
         Current function value: 0.466475
         Iterations 10
                           Logit Regression Results                           
Dep. Variable:              IsDelayed   No. Observations:               200000
Model:                          Logit   Df Residuals:                   199965
Method:                           MLE   Df Model:                           34
Date:                Mon, 09 Mar 2026   Pseudo R-squ.:                  0.1154
Time:                        22:49:43   Log-Likelihood:                -93295.
converged:                       True   LL-Null:                   -1.0547e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept                     -4.3289      0.051    -84.818      0.000     